# Eleições 2026 — votos e abstenção por local de votação
### Presidente (1.º e 2.º turnos) e Deputados, com vínculo aos municípios do IBGE

**Pergunta operacional.** Em cada local de votação e em cada município: quantos eleitores estavam aptos,
quantos compareceram, quantos se abstiveram e como os votos se distribuíram entre os candidatos.

**Fontes oficiais (baixadas pelo próprio notebook):**

| Fonte | Arquivo / API | Uso |
|---|---|---|
| TSE | `eleitorado_local_votacao_2026.zip` | aptos por seção, local, endereço e coordenadas (um registro por seção **e por turno**) |
| TSE | `votacao_secao_2026_BR.zip` | votos para Presidente por seção (1.º e 2.º turnos) |
| TSE | Boletim de Urna, 2.º turno (`bweb_2t_<UF>_*.zip`) | **plano B**: votos do 2.º turno, caso o arquivo acima ainda não tenha sido atualizado |
| TSE | `votacao_secao_2026_<UF>.zip` | votos para Deputado por seção (todos os cargos da UF) |
| TSE | `votacao_candidato_munzona_2026.zip` | votos nominais de cada candidato a Deputado por município e zona (Brasil inteiro) |
| TSE | tabela de municípios do site de resultados | correspondência oficial entre o código TSE e o código IBGE de cada município |
| IBGE | SIDRA, tabela 4709 (Censo 2022); como plano B, o FTP do Censo 2022 | população residente do município |
| IBGE | API de Localidades | plano B do vínculo: código e nome oficial do município |

**Definições (as mesmas dos notebooks anteriores de abstenção):**

| Grandeza | Cálculo |
|---|---|
| Aptos | soma de `QT_ELEITOR_ELEICAO_FEDERAL` das seções do turno |
| Comparecimento | soma de `QT_VOTOS` para Presidente (nominais + brancos + nulos) |
| Abstenção | Aptos − Comparecimento |
| % Abstenção | 100 × Abstenção ÷ Aptos |
| Aptos por 100 habitantes | 100 × Aptos ÷ População do Censo 2022 (indicador de contexto, **não** é taxa de abstenção) |

**Ressalvas metodológicas**
1. O denominador oficial da abstenção é o **cadastro eleitoral do TSE**. O IBGE fornece contexto
   demográfico: população não é eleitorado, porque inclui menores de 16 anos e porque o cadastro
   contém eleitores que mudaram de município sem transferir o título.
2. **Deputados não têm 2.º turno.** O 2.º turno existe apenas para Presidente e Governador.
3. No 2.º turno, o TSE mudou o local de 4.352 seções. Por isso, o local de cada seção é lido do
   cadastro **do próprio turno**.
4. O 2.º turno ocorreu em 25/10/2026. O TSE atualiza o arquivo de votos por seção "ao final de
   cada turno". Por isso, o notebook (a) compara a cópia local com o servidor e baixa de novo se o
   arquivo mudou e (b) se o arquivo por seção ainda não trouxer o 2.º turno, usa o **Boletim de
   Urna do 2.º turno**, conferindo o SHA-512 que o próprio TSE publica.
5. A análise é feita em nível agregado (local e município), não em nível de eleitor, o que traz
   risco de falácia ecológica.

**Como a auditabilidade é garantida:** (a) registro do SHA-256 de cada arquivo baixado;
(b) junções com `validate=` (o pandas interrompe a execução se surgir duplicidade);
(c) `assert` para conferir que os totais fecham; (d) nada é descartado em silêncio: votos sem
local e municípios sem código IBGE vão para a tabela de auditoria; (e) o vínculo TSE→IBGE é
exportado junto com o método usado em cada caso.

## Etapa 1 — Configuração

**Bibliotecas e por que foram escolhidas**

| Biblioteca | Motivo |
|---|---|
| `pandas` | padrão para tabelas em Python. Lê CSV em blocos (os arquivos do TSE têm dezenas de milhões de linhas), agrega com `groupby` e verifica junções com `merge(validate=...)` |
| `numpy` | classificações vetorizadas (`np.select`, `np.where`), muito mais rápidas que laços |
| `requests` | download por *streaming* (grava em disco aos poucos, sem carregar o ZIP inteiro na memória) e consulta leve (`HEAD`) para saber se o arquivo mudou no servidor |
| `time` | espera progressiva entre novas tentativas quando um servidor não responde |
| `zipfile` | lê o CSV direto do ZIP, sem extrair. O arquivo original fica intacto e o hash continua válido |
| `hashlib` | calcula o SHA-256, a "impressão digital" de cada arquivo, que permite provar que outra pessoa usou exatamente os mesmos dados |
| `unicodedata`, `re`, `difflib` | normalizam e comparam nomes de municípios (TSE × IBGE). Fazem parte da biblioteca padrão, então não exigem instalação |

**Variáveis de configuração**

| Variável | Por que existe |
|---|---|
| `TURNOS` | turnos a processar. O 2.º turno é ignorado automaticamente se ainda não tiver sido publicado |
| `UFS_DEPUTADOS` | UFs com deputados por local de votação. Cada arquivo traz todos os cargos de todas as seções da UF (o de SP tem centenas de MB), por isso o padrão começa com uma UF pequena |
| `CARGOS_DEPUTADO` | inclui `DEPUTADO DISTRITAL`, porque no DF ele substitui o estadual |
| `CHAVE_SECAO` / `CHAVE_LOCAL` | o número da seção e o do local **só são únicos dentro da zona eleitoral**, então a zona entra na chave |
| `COL_APTOS` | `QT_ELEITOR_ELEICAO_FEDERAL` = eleitores aptos para a eleição federal. No notebook de abstenção, essa coluna reproduziu o total oficial de 158.745.502 aptos |
| `REFERENCIA_TSE` | totais oficiais usados **apenas para conferência**; não entram em nenhum cálculo |
| `CORRECOES_IBGE` | correções manuais do vínculo TSE→IBGE, para os casos que a etapa 6 marcar como "conferir" |

In [ ]:
import hashlib, json, re, shutil, sys, time, unicodedata, zipfile
from datetime import datetime, timezone
from difflib import get_close_matches
from pathlib import Path

import numpy as np
import pandas as pd
import requests

ANO = 2026
TURNOS = [1, 2]
UFS_DEPUTADOS = ["DF"]            # ex.: ["SP", "RJ", "MG"]
CARGOS_DEPUTADO = ["DEPUTADO FEDERAL", "DEPUTADO ESTADUAL", "DEPUTADO DISTRITAL"]
CORRECOES_IBGE = {}               # {("UF", código_município_TSE): código_IBGE}

PASTA = Path("/content/eleicao2026")
ZIPS, BRUTOS, SAIDA = PASTA / "zips", PASTA / "brutos", PASTA / "csv"
for p in (ZIPS, BRUTOS, SAIDA):
    p.mkdir(parents=True, exist_ok=True)

TSE = "https://cdn.tse.jus.br/estatistica/sead/odsele"
URLS = {
    "eleitorado": f"{TSE}/eleitorado_locais_votacao/eleitorado_local_votacao_{ANO}.zip",
    "presidente": f"{TSE}/votacao_secao/votacao_secao_{ANO}_BR.zip",
    "deputados_munzona": f"{TSE}/votacao_candidato_munzona/votacao_candidato_munzona_{ANO}.zip",
    **{f"secao_{uf}": f"{TSE}/votacao_secao/votacao_secao_{ANO}_{uf}.zip" for uf in UFS_DEPUTADOS},
}
TSE_RESULTADOS = "https://resultados.tse.jus.br/oficial"
CKAN_BOLETIM = f"https://dadosabertos.tse.jus.br/api/3/action/package_show?id=resultados-{ANO}-boletim-de-urna"
URL_IBGE_MUN = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios?view=nivelado"
URL_IBGE_POP = "https://apisidra.ibge.gov.br/values/t/4709/n6/all/v/93/p/2022"
URL_IBGE_FTP_POP = ("https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/"
                    "Agregados_por_Municipio_csv/Agregados_por_municipios_cor_ou_raca_BR.zip")
UA = {"User-Agent": "Mozilla/5.0 (pesquisa academica; Google Colab)"}

CHAVE_SECAO = ["SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"]
CHAVE_URNA = ["SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO_URNA"]
CHAVE_LOCAL = ["SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_LOCAL_VOTACAO"]
COL_APTOS = "QT_ELEITOR_ELEICAO_FEDERAL"
BRANCO_NULO = ["VOTO BRANCO", "VOTO NULO"]
REFERENCIA_TSE = {1: {"APTOS": 158_745_502, "COMPARECIMENTO": 125_275_835},
                  2: {"APTOS": None, "COMPARECIMENTO": None}}   # preencher com o painel oficial do 2.º turno

MANIFESTO, AUDITORIA = [], []

## Etapa 2 — Download e manifesto das fontes

- **`http_get`**: toda requisição usa um tempo curto de conexão (15 s) e até 3 tentativas, com
  espera progressiva. Erros 4xx (arquivo inexistente) não são repetidos.
- **Arquivo atualizado?** Antes de reaproveitar um ZIP já baixado, uma consulta `HEAD` compara o
  tamanho da cópia local com o do servidor. Se o TSE publicou uma versão nova (por exemplo, com o
  2.º turno), o arquivo é baixado de novo. A data `Last-Modified` do servidor vai para o manifesto.
- **Manifesto**: URL, tamanho, SHA-256 e data de cada fonte. Quando o TSE publica um `.sha512` (caso
  do Boletim de Urna), o arquivo é conferido com ele, e uma divergência interrompe a execução.

In [ ]:
def http_get(url, tentativas=3, timeout=(15, 600), **kw):
    for i in range(1, tentativas + 1):
        try:
            r = requests.get(url, headers=UA, timeout=timeout, **kw)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            status = getattr(e.response, "status_code", None)
            if (status and status < 500) or i == tentativas:
                raise
            print(f"  sem resposta ({type(e).__name__}); tentativa {i + 1}/{tentativas} em {5 * i} s: {url}")
            time.sleep(5 * i)


def hash_arquivo(caminho, algoritmo="sha256"):
    h = hashlib.new(algoritmo)
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(8 << 20), b""):
            h.update(bloco)
    return h.hexdigest()


def registrar(url, arquivo, sha256, n_bytes, **extra):
    MANIFESTO.append({"arquivo": arquivo, "url": url, "bytes": n_bytes, "sha256": sha256, **extra,
                      "registrado_em_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")})


def baixar(url, url_sha512=None):
    destino = ZIPS / url.rsplit("/", 1)[-1]
    try:
        h = requests.head(url, headers=UA, timeout=(15, 60), allow_redirects=True)
        tam, modificado = (int(h.headers.get("Content-Length", 0)), h.headers.get("Last-Modified")) if h.ok else (0, None)
    except requests.RequestException:
        tam, modificado = 0, None
    if zipfile.is_zipfile(destino) and tam in (0, destino.stat().st_size):
        print("Cópia local atualizada:", destino.name)
    else:
        print("Baixando", destino.name, "...")
        parcial = destino.with_suffix(".parcial")
        with http_get(url, stream=True) as r, open(parcial, "wb") as f:
            for bloco in r.iter_content(8 << 20):
                f.write(bloco)
        if not zipfile.is_zipfile(parcial):
            raise IOError(f"{destino.name}: download incompleto ou o arquivo não é um ZIP")
        parcial.replace(destino)
    extra = {"last_modified_servidor": modificado}
    if url_sha512:
        oficial = re.search(r"[0-9a-fA-F]{128}", http_get(url_sha512, timeout=(15, 60)).text).group(0).lower()
        extra["sha512_confere_com_tse"] = hash_arquivo(destino, "sha512") == oficial
        assert extra["sha512_confere_com_tse"], f"{destino.name}: SHA-512 diferente do publicado pelo TSE"
    registrar(url, destino.name, hash_arquivo(destino), destino.stat().st_size, **extra)
    return zipfile.ZipFile(destino)


def baixar_json(url, nome, **kw):
    r = http_get(url, **kw)
    (BRUTOS / nome).write_bytes(r.content)
    registrar(url, f"brutos/{nome}", hashlib.sha256(r.content).hexdigest(), len(r.content))
    return json.loads(r.content.decode("utf-8-sig"))


ZF = {nome: baixar(url) for nome, url in URLS.items()}
pd.DataFrame(MANIFESTO)

## Etapa 3 — Função de leitura

Os CSV do TSE usam `;` como separador e a codificação `latin1`. A função lê **somente as colunas
necessárias**, em blocos de 1 milhão de linhas, e aplica o filtro (cargo, turno) a cada bloco.
Isso reduz o uso de memória no Colab. Se uma coluna obrigatória não existir, a execução para com
uma mensagem clara, em vez de produzir uma tabela incompleta. As coordenadas são lidas como texto
porque usam vírgula decimal.

In [ ]:
def ler(zf, colunas, opcionais=(), filtro=None, texto=()):
    nomes = [n for n in zf.namelist() if n.lower().endswith(".csv")]
    nomes = [n for n in nomes if n.upper().endswith("_BRASIL.CSV")] or nomes   # prefere o arquivo nacional
    partes = []
    for nome in nomes:
        with zf.open(nome) as f:
            cab = pd.read_csv(f, sep=";", encoding="latin1", nrows=0).columns
        faltam = [c for c in colunas if c not in cab and c not in opcionais]
        if faltam:
            raise KeyError(f"{nome}: colunas ausentes {faltam}")
        usar = [c for c in colunas if c in cab]
        with zf.open(nome) as f:
            for bloco in pd.read_csv(f, sep=";", encoding="latin1", usecols=usar, chunksize=1_000_000,
                                     dtype={c: str for c in texto if c in usar}):
                partes.append(filtro(bloco) if filtro else bloco)
    print(f"  {Path(zf.filename).name}: {len(nomes)} CSV lido(s)")
    return pd.concat(partes, ignore_index=True)

## Etapa 4 — Cadastro de seções (aptos e locais)

| Variável | Por que foi escolhida |
|---|---|
| `NR_TURNO` | o cadastro tem uma linha por seção **em cada turno**. Sem esse filtro, os aptos dobram, e é esse cadastro por turno que registra as 4.352 seções que mudaram de local |
| `QT_ELEITOR_ELEICAO_FEDERAL` | eleitores aptos da seção (denominador da abstenção) |
| `NR_LOCAL_VOTACAO`, `NM_LOCAL_VOTACAO`, `DS_ENDERECO`, `NM_BAIRRO` | identificam o ponto de votação (escola, ginásio etc.) |
| `NR_LATITUDE`, `NR_LONGITUDE` | coordenadas do ponto. O valor −1 indica ausência de coordenada e vira vazio (`NaN`) |
| `NR_SECAO_PRINCIPAL` | seções pequenas são **agregadas** a uma seção principal e votam na urna dela (ver etapa 5) |

In [ ]:
eleitorado = ler(
    ZF["eleitorado"],
    ["NR_TURNO", *CHAVE_SECAO, "NM_MUNICIPIO", "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO", "DS_ENDERECO",
     "NM_BAIRRO", "NR_LATITUDE", "NR_LONGITUDE", "NR_SECAO_PRINCIPAL", COL_APTOS],
    opcionais=["DS_ENDERECO", "NM_BAIRRO", "NR_SECAO_PRINCIPAL"],
    texto=["NR_LATITUDE", "NR_LONGITUDE"],
)
for c in ["NR_LATITUDE", "NR_LONGITUDE"]:
    eleitorado[c] = pd.to_numeric(eleitorado[c].str.replace(",", ".", regex=False), errors="coerce")
    eleitorado.loc[eleitorado[c] == -1, c] = np.nan

print(eleitorado.groupby("NR_TURNO")[COL_APTOS].agg(secoes="size", aptos="sum"))

## Etapa 5 — Da seção à urna e da urna ao local de votação

- **Aptos** vêm do cadastro e **votos** vêm do boletim de urna. Os dois são somados de forma
  independente e depois reunidos pelo local de votação.
- Uma **seção agregada** não tem urna própria: seus eleitores votam na urna da seção principal.
  Por isso, os aptos dela são somados à seção principal (`NR_SECAO_URNA`) antes da junção com
  os votos. Sem essa correção, a seção agregada aparece com 100% de abstenção e a principal
  aparece com comparecimento acima do número de aptos.
- O local da urna é o local da seção principal no cadastro **do turno**.
- Votos ou aptos que não encontram um local **não são descartados em silêncio**: eles são
  contados e vão para a tabela de auditoria.

In [ ]:
def eleitorado_turno(turno):
    el = eleitorado[eleitorado["NR_TURNO"] == turno].drop(columns="NR_TURNO")
    assert not el.duplicated(CHAVE_SECAO).any(), "Seção repetida no cadastro do turno"
    return el if len(el) else None


def base_urnas(el):
    p = pd.to_numeric(el.get("NR_SECAO_PRINCIPAL", el["NR_SECAO"]), errors="coerce")
    el = el.assign(NR_SECAO_URNA=p.where(p > 0, el["NR_SECAO"]).astype("int64"))
    urnas = el.groupby(CHAVE_URNA, as_index=False).agg(APTOS=(COL_APTOS, "sum"), QT_SECOES=("NR_SECAO", "size"))
    local = el[CHAVE_SECAO + ["NR_LOCAL_VOTACAO"]].rename(columns={"NR_SECAO": "NR_SECAO_URNA"})
    return urnas.merge(local, on=CHAVE_URNA, how="left", validate="one_to_one")


def aptos_por_local(urnas):
    sem = urnas["NR_LOCAL_VOTACAO"].isna()
    t = (urnas[~sem].astype({"NR_LOCAL_VOTACAO": "int64"}).groupby(CHAVE_LOCAL, as_index=False)
         .agg(QT_URNAS=("NR_SECAO_URNA", "size"), QT_SECOES=("QT_SECOES", "sum"), APTOS=("APTOS", "sum")))
    return t, int(urnas.loc[sem, "APTOS"].sum())


def votos_por_local(votos, urnas, por):
    v = votos.rename(columns={"NR_SECAO": "NR_SECAO_URNA"}).merge(
        urnas[CHAVE_URNA + ["NR_LOCAL_VOTACAO"]], on=CHAVE_URNA, how="left", validate="many_to_one")
    sem = v["NR_LOCAL_VOTACAO"].isna()
    t = (v[~sem].astype({"NR_LOCAL_VOTACAO": "int64"})
         .groupby(CHAVE_LOCAL + por, as_index=False)["QT_VOTOS"].sum())
    return t, int(v.loc[sem, "QT_VOTOS"].sum())


def cadastro_locais(el):
    cols = CHAVE_LOCAL + ["NM_MUNICIPIO", "NM_LOCAL_VOTACAO", "DS_ENDERECO", "NM_BAIRRO",
                          "NR_LATITUDE", "NR_LONGITUDE"]
    return el[[c for c in cols if c in el]].drop_duplicates(CHAVE_LOCAL)


def percentuais(df, candidatos):
    validos = df["VOTOS_VALIDOS"].where(df["VOTOS_VALIDOS"] > 0)
    df["PCT_ABSTENCAO"] = (100 * df["ABSTENCAO"] / df["APTOS"].where(df["APTOS"] > 0)).round(2)
    for c in candidatos:
        df[f"PCT_{c}"] = (100 * df[c] / validos).round(2)
    return df

## Etapa 6 — Vínculo com o IBGE (código e população do município)

**Por que deu `ConnectTimeout` na versão anterior.** A conexão com `servicodados.ibge.gov.br`
sequer foi aberta: o servidor não respondeu em 120 s. É uma falha de rede do lado do IBGE, não do
código. A causa mais provável é que as APIs do IBGE ficam instáveis ou não respondem a acessos
vindos de fora do Brasil, e o Colab roda em data centers no exterior. A correção tem três partes:

1. **Vínculo TSE → IBGE pela tabela oficial do próprio TSE.** O site de resultados publica, para
   cada município, o código TSE (`cd`) e o código IBGE (`cdi`). Essa correspondência é exata,
   dispensa a API de Localidades e elimina a comparação de nomes. O identificador da eleição é
   descoberto no catálogo `ele-c.json`, sem precisar ser digitado.
   *Plano B:* API de Localidades do IBGE, com comparação de nomes (exata e, depois, aproximada).
2. **População:** primeiro o SIDRA (tabela 4709, variável 93, Censo 2022). *Plano B:* o arquivo do
   Censo 2022 no FTP do IBGE (o mesmo servidor que funcionou na análise de cor/raça). Nesse caso, a
   população é a soma das cinco categorias de cor ou raça, `V01317` a `V01321`. O total nacional é
   impresso para conferência: deve ficar em torno de 203,1 milhões.
3. **Tempo de conexão de 15 s com novas tentativas**, em vez de uma única espera de 120 s.

A fonte efetivamente usada fica registrada na auditoria (`FONTE_VINCULO`, `FONTE_POPULACAO`), e as
respostas brutas ficam em `brutos/`, com SHA-256 no manifesto.

In [ ]:
def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"[^A-Z0-9]", "", s)


def vinculo_tabela_tse():
    catalogo = http_get(f"{TSE_RESULTADOS}/comum/config/ele-c.json", timeout=(15, 60)).text
    for cd in dict.fromkeys(re.findall(r'"cd"\s*:\s*"(\d+)"', catalogo)):
        url = f"{TSE_RESULTADOS}/ele{ANO}/{cd}/config/mun-e{cd.zfill(6)}-cm.json"
        try:
            cfg = baixar_json(url, f"tse_municipios_e{cd}.json", tentativas=1, timeout=(15, 60))
        except requests.RequestException:
            continue
        return pd.DataFrame([(uf["cd"].upper(), int(m["cd"]), m.get("cdi")) for uf in cfg["abr"]
                             for m in uf.get("mu", [])], columns=["SG_UF", "CD_MUNICIPIO", "CD_IBGE"])
    raise RuntimeError(f"tabela de municípios de {ANO} não encontrada no site de resultados")


def vinculo_nome_ibge():
    ibge = (pd.DataFrame(baixar_json(URL_IBGE_MUN, "ibge_municipios.json", tentativas=2, timeout=(15, 60)))
            .rename(columns={"municipio-id": "CD_IBGE", "municipio-nome": "NOME", "UF-sigla": "SG_UF"}))
    ibge["CHAVE"] = ibge["NOME"].map(normalizar)
    linhas = []
    for uf, cod, nome in mun_tse[["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO"]].itertuples(index=False):
        da_uf, chave = ibge[ibge["SG_UF"] == uf], normalizar(nome)
        achado = [chave] if chave in set(da_uf["CHAVE"]) else get_close_matches(chave, da_uf["CHAVE"].tolist(), n=1, cutoff=0.8)
        metodo = "nome idêntico" if achado == [chave] else "nome aproximado (conferir)" if achado else "sem correspondência"
        linhas.append((uf, cod, da_uf.loc[da_uf["CHAVE"] == achado[0], "CD_IBGE"].iat[0] if achado else None, metodo))
    return pd.DataFrame(linhas, columns=["SG_UF", "CD_MUNICIPIO", "CD_IBGE", "METODO_VINCULO"])


def populacao_sidra():
    js = baixar_json(URL_IBGE_POP, "ibge_sidra_4709.json", tentativas=2, timeout=(15, 120))
    col = next(k for k, v in js[0].items() if v == "Município (Código)")
    return pd.DataFrame({"CD_IBGE": [int(r[col]) for r in js[1:]],
                         "POP_CENSO_2022": pd.to_numeric([r["V"] for r in js[1:]], errors="coerce")})


def populacao_ftp_censo():
    zf = baixar(URL_IBGE_FTP_POP)
    with zf.open(next(n for n in zf.namelist() if n.lower().endswith(".csv"))) as f:
        df = pd.read_csv(f, sep=";", encoding="latin1")
    cor = ["V01317", "V01318", "V01319", "V01320", "V01321"]
    return pd.DataFrame({"CD_IBGE": df["CD_MUN"].astype("int64"),
                         "POP_CENSO_2022": df[cor].apply(pd.to_numeric, errors="coerce").sum(axis=1)})


def primeira_fonte(*fontes):
    for fonte in fontes:
        try:
            return fonte(), fonte.__name__
        except Exception as e:
            print(f"  {fonte.__name__} indisponível -> {type(e).__name__}: {str(e)[:150]}")
    raise RuntimeError("Nenhuma fonte respondeu. Aguarde alguns minutos e execute a célula novamente.")


mun_tse = (eleitorado.loc[eleitorado["SG_UF"] != "ZZ", ["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO"]]
           .drop_duplicates(["SG_UF", "CD_MUNICIPIO"]))
correspondencia, FONTE_VINCULO = primeira_fonte(vinculo_tabela_tse, vinculo_nome_ibge)
populacao, FONTE_POPULACAO = primeira_fonte(populacao_sidra, populacao_ftp_censo)

vinculo = mun_tse.merge(correspondencia, on=["SG_UF", "CD_MUNICIPIO"], how="left", validate="one_to_one")
vinculo["CD_IBGE"] = pd.to_numeric(vinculo["CD_IBGE"], errors="coerce").astype("Int64")
if "METODO_VINCULO" not in vinculo:
    vinculo["METODO_VINCULO"] = "tabela oficial TSE"
vinculo.loc[vinculo["CD_IBGE"].isna(), "METODO_VINCULO"] = "sem correspondência"
for (uf, cod), cd_ibge in CORRECOES_IBGE.items():
    alvo = (vinculo["SG_UF"] == uf) & (vinculo["CD_MUNICIPIO"] == cod)
    vinculo.loc[alvo, ["CD_IBGE", "METODO_VINCULO"]] = [cd_ibge, "correção manual"]
vinculo.loc[vinculo["CD_IBGE"].notna() & vinculo.duplicated("CD_IBGE", keep=False), "METODO_VINCULO"] += " | CÓDIGO IBGE REPETIDO"

vinculo = vinculo.merge(populacao, on="CD_IBGE", how="left", validate="many_to_one")
vinculo["POP_CENSO_2022"] = vinculo["POP_CENSO_2022"].astype("Int64")

print(f"Vínculo: {FONTE_VINCULO} | População: {FONTE_POPULACAO}")
print(vinculo["METODO_VINCULO"].value_counts().to_string())
print(f"População total vinculada: {int(vinculo['POP_CENSO_2022'].sum()):,}".replace(",", "."))
pendentes = vinculo[~vinculo["METODO_VINCULO"].isin(["tabela oficial TSE", "nome idêntico"])]
print(pendentes.to_string() if len(pendentes) else "Nenhum município pendente de conferência.")
AUDITORIA.append({"TABELA": "vinculo_ibge", "FONTE_VINCULO": FONTE_VINCULO, "FONTE_POPULACAO": FONTE_POPULACAO,
                  "MUNICIPIOS_TSE": len(vinculo), "MUNICIPIOS_SEM_CD_IBGE": int(vinculo["CD_IBGE"].isna().sum()),
                  "MUNICIPIOS_SEM_POPULACAO": int(vinculo["POP_CENSO_2022"].isna().sum()),
                  "POPULACAO_TOTAL": int(vinculo["POP_CENSO_2022"].sum())})

MUN_IBGE = vinculo[["SG_UF", "CD_MUNICIPIO", "CD_IBGE", "POP_CENSO_2022"]]

## Etapa 7 — Presidente: tabelas por local de votação e por município (1.º e 2.º turnos)

**Tabela por local** (`presidente_2026_T<turno>_por_local.csv`): uma linha por ponto de votação,
com endereço, coordenadas, código IBGE e população do município, número de urnas e seções, aptos,
votos de cada candidato, brancos, nulos, válidos, comparecimento, abstenção e percentuais.

**Tabela por município**: soma dos locais, acrescida de `APTOS_POR_100_HAB` e
`ABSTENCAO_POR_100_HAB`, ambos calculados sobre a população do Censo.

`COMPARECIMENTO_MAIOR_QUE_APTOS` sinaliza locais em que votaram mais pessoas do que o número de
aptos do cadastro. Isso pode acontecer legitimamente, por exemplo com o voto em trânsito ou com
mesários que votam onde trabalham. A coluna existe para inspeção, e esses locais não são excluídos.

**Origem dos votos de cada turno (`FONTE_VOTOS`)**
- Principal: `votacao_secao_2026_BR.zip`.
- Plano B, caso esse arquivo ainda não traga o turno: o **Boletim de Urna** (BU), o registro
  oficial de cada urna. Os endereços dos ZIPs (`bweb_2t_<UF>_<data>.zip`) são obtidos na API do
  Portal de Dados Abertos. Se uma UF tiver mais de uma geração, vale a mais recente. Cada ZIP é
  conferido com o `.sha512` do TSE.
- No BU, o cargo está em `DS_CARGO_PERGUNTA`, e os votos branco e nulo aparecem como "Branco" e
  "Nulo". Os nomes são padronizados para "VOTO BRANCO" e "VOTO NULO", como no arquivo por seção.
- **Cobertura:** se alguma UF ficar sem votos (por exemplo, BU ainda não publicado), ela é
  **excluída** da tabela do turno e listada na auditoria. Sem isso, essa UF apareceria com
  abstenção de 100%.

In [ ]:
def votos_boletim_urna(turno):
    recursos = baixar_json(CKAN_BOLETIM, "ckan_boletim_de_urna.json", timeout=(15, 60))["result"]["resources"]
    urls, mais_recente = {r["url"] for r in recursos}, {}
    for u in urls:
        m = re.search(rf"bweb_{turno}t_([A-Z]{{2}})_(\d{{12}})\.zip$", u)
        if m:
            data = datetime.strptime(m.group(2), "%d%m%Y%H%M")
            if m.group(1) not in mais_recente or data > mais_recente[m.group(1)][0]:
                mais_recente[m.group(1)] = (data, u)
    print(f"  Boletins de Urna do {turno}.º turno publicados: {len(mais_recente)} UF(s)")
    partes = [ler(baixar(u, u + ".sha512" if u + ".sha512" in urls else None),
                  ["NR_TURNO", "DS_CARGO_PERGUNTA", *CHAVE_SECAO, "NM_VOTAVEL", "QT_VOTOS"],
                  filtro=lambda b: b[b["DS_CARGO_PERGUNTA"].str.upper().str.strip() == "PRESIDENTE"]
                  .drop(columns="DS_CARGO_PERGUNTA"))
              for _, u in sorted(mais_recente.values())]
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


votos_pres = ler(
    ZF["presidente"], ["NR_TURNO", "DS_CARGO", *CHAVE_SECAO, "NM_VOTAVEL", "QT_VOTOS"],
    filtro=lambda b: b[b["DS_CARGO"].str.upper().str.strip() == "PRESIDENTE"].drop(columns="DS_CARGO"))
FONTE_VOTOS = {int(t): f"votacao_secao_{ANO}_BR" for t in votos_pres["NR_TURNO"].unique()}

for turno in [t for t in TURNOS if t not in FONTE_VOTOS]:
    print(f"votacao_secao_{ANO}_BR.zip ainda não traz o {turno}.º turno -> usando o Boletim de Urna")
    try:
        bu = votos_boletim_urna(turno)
    except Exception as e:
        print(f"  Boletim de Urna indisponível ({type(e).__name__}: {str(e)[:150]})")
        continue
    if len(bu):
        votos_pres, FONTE_VOTOS[turno] = pd.concat([votos_pres, bu], ignore_index=True), "boletim_de_urna"

votos_pres["NM_VOTAVEL"] = (votos_pres["NM_VOTAVEL"].str.upper().str.strip()
                            .replace({"BRANCO": "VOTO BRANCO", "NULO": "VOTO NULO"}))
print("Origem dos votos por turno:", FONTE_VOTOS)

pres_local = {}
for turno in TURNOS:
    el, votos = eleitorado_turno(turno), votos_pres[votos_pres["NR_TURNO"] == turno]
    if el is None or votos.empty:
        print(f"\n{turno}.º turno: votos não encontrados em nenhuma fonte do TSE — etapa ignorada.")
        continue
    ufs_sem_votos = sorted(set(el["SG_UF"]) - set(votos["SG_UF"]))
    if ufs_sem_votos:
        print(f"ATENÇÃO {turno}.º turno: UFs sem votos excluídas da tabela: {ufs_sem_votos}")
        el = el[~el["SG_UF"].isin(ufs_sem_votos)]

    urnas = base_urnas(el)
    aptos, aptos_sem_local = aptos_por_local(urnas)
    vl, votos_sem_local = votos_por_local(votos, urnas, ["NM_VOTAVEL"])
    wide = vl.pivot_table(index=CHAVE_LOCAL, columns="NM_VOTAVEL", values="QT_VOTOS", aggfunc="sum", fill_value=0)
    candidatos = [c for c in wide.columns if c not in BRANCO_NULO]
    comparecimento = wide.sum(axis=1)
    wide["VOTOS_VALIDOS"] = wide[candidatos].sum(axis=1)
    wide["COMPARECIMENTO"] = comparecimento
    contagens = list(wide.columns)

    local = (cadastro_locais(el).merge(aptos, on=CHAVE_LOCAL, how="right", validate="one_to_one")
             .merge(wide.reset_index(), on=CHAVE_LOCAL, how="left", validate="one_to_one")
             .merge(MUN_IBGE, on=["SG_UF", "CD_MUNICIPIO"], how="left", validate="many_to_one"))
    local[contagens] = local[contagens].fillna(0).astype("int64")
    local["ABSTENCAO"] = local["APTOS"] - local["COMPARECIMENTO"]
    local["COMPARECIMENTO_MAIOR_QUE_APTOS"] = local["COMPARECIMENTO"] > local["APTOS"]
    local = percentuais(local, candidatos)

    somar = ["QT_URNAS", "QT_SECOES", "APTOS", *contagens, "ABSTENCAO"]
    mun = (local.groupby(["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO", "CD_IBGE", "POP_CENSO_2022"], dropna=False)
           .agg(QT_LOCAIS=("NR_LOCAL_VOTACAO", "size"), **{c: (c, "sum") for c in somar}).reset_index())
    mun = percentuais(mun, candidatos)
    mun["APTOS_POR_100_HAB"] = (100 * mun["APTOS"] / mun["POP_CENSO_2022"]).round(2)
    mun["ABSTENCAO_POR_100_HAB"] = (100 * mun["ABSTENCAO"] / mun["POP_CENSO_2022"]).round(2)
    local, mun = (d.astype({"CD_IBGE": "Int64", "POP_CENSO_2022": "Int64"}) for d in (local, mun))

    # ---- auditoria: os totais precisam fechar com os arquivos de origem ----------
    aptos_total, comp_total = int(el[COL_APTOS].sum()), int(votos["QT_VOTOS"].sum())
    assert local["APTOS"].sum() + aptos_sem_local == aptos_total, "aptos não fecham"
    assert local["COMPARECIMENTO"].sum() + votos_sem_local == comp_total, "comparecimento não fecha"
    assert mun["APTOS"].sum() == local["APTOS"].sum() and not local.duplicated(CHAVE_LOCAL).any()
    ref = REFERENCIA_TSE.get(turno, {})
    AUDITORIA.append({
        "TABELA": f"presidente_T{turno}", "FONTE_VOTOS": FONTE_VOTOS[turno],
        "UFS_EXCLUIDAS_SEM_VOTOS": ", ".join(ufs_sem_votos) or None, "APTOS_ARQUIVO": aptos_total, "APTOS_OFICIAL_TSE": ref.get("APTOS"),
        "COMPARECIMENTO_ARQUIVO": comp_total, "COMPARECIMENTO_OFICIAL_TSE": ref.get("COMPARECIMENTO"),
        "ABSTENCAO": aptos_total - comp_total, "PCT_ABSTENCAO": round(100 * (aptos_total - comp_total) / aptos_total, 2),
        "APTOS_SEM_LOCAL": aptos_sem_local, "VOTOS_SEM_LOCAL": votos_sem_local, "LOCAIS": len(local),
        "LOCAIS_COMPARECIMENTO_MAIOR_QUE_APTOS": int(local["COMPARECIMENTO_MAIOR_QUE_APTOS"].sum()),
        "MUNICIPIOS_SEM_CD_IBGE": int(mun.loc[mun["SG_UF"] != "ZZ", "CD_IBGE"].isna().sum())})
    print(f"\n{turno}.º turno:", {k: v for k, v in AUDITORIA[-1].items() if k != "TABELA"})

    local.to_csv(SAIDA / f"presidente_{ANO}_T{turno}_por_local.csv", index=False, encoding="utf-8-sig")
    mun.to_csv(SAIDA / f"presidente_{ANO}_T{turno}_por_municipio.csv", index=False, encoding="utf-8-sig")
    pres_local[turno] = local

## Etapa 8 — Deputados: votos por candidato e município (Brasil inteiro)

Usamos `votacao_candidato_munzona`, um arquivo único para todo o país e muito mais leve que os
arquivos por seção.

| Variável | Por que foi escolhida |
|---|---|
| `QT_VOTOS_NOMINAIS` | votos dados ao número do candidato. Os votos de legenda ficam em `votacao_partido_munzona` e entram no cálculo do quociente partidário, não no desempenho individual |
| `DS_SIT_TOT_TURNO` | situação final do candidato (eleito por QP, eleito por média, suplente, não eleito) |
| `NM_URNA_CANDIDATO`, `SG_PARTIDO` | nome de urna e partido, para leitura humana da tabela |

As zonas são somadas para formar o município, e o total nacional de votos é conferido antes e
depois da agregação.

In [ ]:
dep_mz = ler(
    ZF["deputados_munzona"],
    ["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO", "NR_ZONA", "DS_CARGO", "NR_CANDIDATO", "NM_URNA_CANDIDATO",
     "SG_PARTIDO", "DS_SIT_TOT_TURNO", "QT_VOTOS_NOMINAIS"],
    opcionais=["NM_URNA_CANDIDATO", "SG_PARTIDO", "DS_SIT_TOT_TURNO"],
    filtro=lambda b: b[b["DS_CARGO"].str.upper().str.strip().isin(CARGOS_DEPUTADO)])

chave_dep = [c for c in ["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO", "DS_CARGO", "NR_CANDIDATO",
                         "NM_URNA_CANDIDATO", "SG_PARTIDO", "DS_SIT_TOT_TURNO"] if c in dep_mz]
dep_mun = (dep_mz.groupby(chave_dep, dropna=False, as_index=False)["QT_VOTOS_NOMINAIS"].sum()
           .merge(MUN_IBGE, on=["SG_UF", "CD_MUNICIPIO"], how="left", validate="many_to_one"))
assert dep_mun["QT_VOTOS_NOMINAIS"].sum() == dep_mz["QT_VOTOS_NOMINAIS"].sum(), "votos de deputados não fecham"

AUDITORIA.append({"TABELA": "deputados_por_municipio", "LINHAS": len(dep_mun),
                  "VOTOS_NOMINAIS": int(dep_mun["QT_VOTOS_NOMINAIS"].sum()),
                  "MUNICIPIOS_SEM_CD_IBGE": int(dep_mun.loc[dep_mun["SG_UF"] != "ZZ", "CD_IBGE"].isna().sum())})
print(dep_mun.groupby("DS_CARGO")["QT_VOTOS_NOMINAIS"].sum())
dep_mun.to_csv(SAIDA / f"deputados_{ANO}_por_municipio.csv", index=False, encoding="utf-8-sig")

## Etapa 9 — Deputados por local de votação (UFs de `UFS_DEPUTADOS`)

Aplicamos a mesma regra urna → local da etapa 5, com o cadastro do 1.º turno. A tabela é **longa**
(uma linha por local × votável), porque uma UF grande tem milhares de candidatos e não caberia
em colunas.

| Variável criada | Regra |
|---|---|
| `TIPO_VOTO` | `NR_VOTAVEL` 95 = branco; 96 = nulo; 2 dígitos = legenda (número do partido); demais = nominal |
| `COMPARECIMENTO_CARGO` | soma de todos os votos do cargo no local |
| `DIF_COMPARECIMENTO_VS_PRESIDENTE` | comparecimento do cargo menos o de Presidente. O esperado é 0. Diferenças indicam eleitores habilitados apenas para Presidente, como os que votaram em trânsito fora da UF |

Os dados de contexto do local (endereço, coordenadas, aptos, abstenção e código IBGE) vêm da
tabela de Presidente do 1.º turno.

In [ ]:
urnas_t1 = base_urnas(eleitorado_turno(1))
contexto = pres_local[1][[c for c in CHAVE_LOCAL + ["NM_MUNICIPIO", "CD_IBGE", "NM_LOCAL_VOTACAO", "DS_ENDERECO",
                          "NM_BAIRRO", "NR_LATITUDE", "NR_LONGITUDE", "APTOS", "COMPARECIMENTO", "ABSTENCAO",
                          "PCT_ABSTENCAO"] if c in pres_local[1]]]

for uf in UFS_DEPUTADOS:
    v = ler(ZF[f"secao_{uf}"], ["NR_TURNO", "DS_CARGO", *CHAVE_SECAO, "NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"],
            filtro=lambda b: b[(b["NR_TURNO"] == 1) & b["DS_CARGO"].str.upper().str.strip()
                               .isin(CARGOS_DEPUTADO)].drop(columns="NR_TURNO"))
    v["DS_CARGO"] = v["DS_CARGO"].str.upper().str.strip()
    dl, sem_local = votos_por_local(v, urnas_t1, ["DS_CARGO", "NR_VOTAVEL", "NM_VOTAVEL"])
    dl["TIPO_VOTO"] = np.select([dl["NR_VOTAVEL"] == 95, dl["NR_VOTAVEL"] == 96, dl["NR_VOTAVEL"] < 100],
                                ["BRANCO", "NULO", "LEGENDA"], "NOMINAL")
    dl["COMPARECIMENTO_CARGO"] = dl.groupby(CHAVE_LOCAL + ["DS_CARGO"])["QT_VOTOS"].transform("sum")
    dl = dl.merge(contexto, on=CHAVE_LOCAL, how="left", validate="many_to_one")
    dl["DIF_COMPARECIMENTO_VS_PRESIDENTE"] = dl["COMPARECIMENTO_CARGO"] - dl["COMPARECIMENTO"]

    assert dl["QT_VOTOS"].sum() + sem_local == v["QT_VOTOS"].sum(), f"{uf}: votos não fecham"
    por_local = dl.drop_duplicates(CHAVE_LOCAL + ["DS_CARGO"])
    AUDITORIA.append({"TABELA": f"deputados_por_local_{uf}", "LINHAS": len(dl), "VOTOS_ARQUIVO": int(v["QT_VOTOS"].sum()),
                      "VOTOS_SEM_LOCAL": sem_local, "LOCAIS": dl[CHAVE_LOCAL].drop_duplicates().shape[0],
                      "LOCAIS_CARGO_COM_DIF_VS_PRESIDENTE": int((por_local["DIF_COMPARECIMENTO_VS_PRESIDENTE"] != 0).sum())})
    print(uf, {k: v_ for k, v_ in AUDITORIA[-1].items() if k != "TABELA"})
    dl.to_csv(SAIDA / f"deputados_{ANO}_por_local_{uf}.csv", index=False, encoding="utf-8-sig")

## Etapa 10 — Auditoria, manifesto e download

| Arquivo | Conteúdo |
|---|---|
| `presidente_2026_T1_por_local.csv` / `_T2_` | uma linha por ponto de votação |
| `presidente_2026_T1_por_municipio.csv` / `_T2_` | uma linha por município, com o código e a população do IBGE |
| `deputados_2026_por_municipio.csv` | candidato × município, para o Brasil inteiro |
| `deputados_2026_por_local_<UF>.csv` | local × votável, para as UFs escolhidas |
| `vinculo_municipios_tse_ibge.csv` | correspondência TSE→IBGE, com o método usado em cada caso |
| `auditoria.csv` | totais, diferenças em relação ao TSE oficial e itens que pedem inspeção |
| `manifesto_fontes.csv` | URL, tamanho, SHA-256, `Last-Modified` do servidor, conferência do SHA-512 do TSE (quando houver), versões do Python e do pandas |

As respostas brutas das APIs (TSE e IBGE) ficam em `brutos/` e também entram no pacote `.zip`.

In [ ]:
pd.DataFrame(MANIFESTO).assign(python=sys.version.split()[0], pandas=pd.__version__).to_csv(
    SAIDA / "manifesto_fontes.csv", index=False, encoding="utf-8-sig")
vinculo.to_csv(SAIDA / "vinculo_municipios_tse_ibge.csv", index=False, encoding="utf-8-sig")
auditoria = pd.DataFrame(AUDITORIA)
for campo in ["APTOS", "COMPARECIMENTO"]:
    if f"{campo}_OFICIAL_TSE" in auditoria:
        auditoria[f"DIF_{campo}_VS_OFICIAL"] = auditoria[f"{campo}_ARQUIVO"] - auditoria[f"{campo}_OFICIAL_TSE"]
auditoria.to_csv(SAIDA / "auditoria.csv", index=False, encoding="utf-8-sig")
print(auditoria.T.to_string())

for arq in sorted(SAIDA.iterdir()):
    print(f"{arq.name:45s} {arq.stat().st_size / 1e6:8.2f} MB")

shutil.copytree(BRUTOS, SAIDA / "brutos", dirs_exist_ok=True)
pacote = shutil.make_archive(str(PASTA / f"eleicao{ANO}_csv"), "zip", SAIDA)
try:
    from google.colab import files
    files.download(pacote)
except ImportError:
    print("Fora do Colab. Pacote salvo em:", pacote)

## Como um terceiro audita este resultado

1. Execute o notebook e compare os `sha256` do `manifesto_fontes.csv` com os registrados aqui.
   Se forem iguais, os dados de partida são os mesmos.
2. Em `auditoria.csv`, `DIF_APTOS_VS_OFICIAL` e `DIF_COMPARECIMENTO_VS_OFICIAL` devem ser 0, ou
   ter uma diferença explicada. Para o 2.º turno, preencha `REFERENCIA_TSE[2]` com os números do
   painel oficial de resultados.
3. Confira `FONTE_VOTOS`, `FONTE_VINCULO` e `FONTE_POPULACAO` na auditoria. Se o 2.º turno veio do
   Boletim de Urna, `sha512_confere_com_tse` deve ser `True` em todos os arquivos `bweb_2t_*`.
   `UFS_EXCLUIDAS_SEM_VOTOS` deve estar vazio. Se não estiver, execute de novo quando o TSE concluir
   a publicação.
4. `VOTOS_SEM_LOCAL` e `APTOS_SEM_LOCAL` mostram o que não pôde ser atribuído a um ponto de votação.
   Esses valores não somem: os `assert` garantem que *atribuído + não atribuído = total do arquivo*.
5. Em `vinculo_municipios_tse_ibge.csv`, filtre `METODO_VINCULO` diferente de "tabela oficial TSE" e de "nome idêntico",
   confira os casos e, se necessário, registre a correção em `CORRECOES_IBGE`.
6. Para conferir manualmente um local, filtre `presidente_2026_T1_por_local.csv` por zona e local
   e compare com o boletim de urna das seções correspondentes.